# Quickstart

A basic introduction to `atlas-local-lib-py`: create a MongoDB Atlas Local
deployment, inspect its state, retrieve its connection string, and delete it
when finished.

Requires a Docker daemon on the machine running this kernel.

In [ ]:
%pip install atlas-local-lib-py

## Create a deployment

`get_or_create` is the notebook-friendly entry point: re-running this cell
returns the deployment that already exists instead of failing or creating a
second one. The first run pulls the image, so it takes a while.

In [ ]:
from atlas_local import LocalDeployment

NAME = "quickstart"

deployment = LocalDeployment.get_or_create(name=NAME)
deployment

## Inspect its state

A deployment is running as soon as it is created.

In [ ]:
print("name:         ", deployment.name)
print("container id: ", deployment.container_id[:12])
print("state:        ", deployment.state)
print("image:        ", deployment.image)
print("image tag:    ", deployment.image_tag)
print("mongodb:      ", deployment.mongodb_version)
print("port binding: ", deployment.port_bindings)

It also shows up in the list of local deployments.

In [ ]:
[(other.name, other.state) for other in LocalDeployment.list()]

## Get the connection string

The connection string points at the port Docker published on the host. It only exists while
the deployment is running, so a stopped or paused deployment has to be started
or unpaused first.

In [ ]:
connection_string = deployment.connection_string()
connection_string

## Connect from Python

The connection string returned by this library can be used with any MongoDB-compatible
driver, such as `PyMongo`. `PyMongo` is not a dependency of this library, so
it must be installed separately.

In [ ]:
%pip install pymongo

In [ ]:
import pymongo

client = pymongo.MongoClient(connection_string)
client.quickstart.movies.insert_one({"title": "Local Atlas", "year": 2026})

list(client.quickstart.movies.find({}, {"_id": 0}))

## Delete the deployment

Deleting the deployment removes its container and all data stored in it. To keep it available between notebook sessions, leave this cell unexecuted. The `get_or_create` call in the first cell will reuse it.


In [ ]:
client.close()
deployment.delete()

[other.name for other in LocalDeployment.list()]